<div class="blog-language-switch" role="group" aria-label="Article language"><span aria-current="page">English</span><a href="/ipynb/zh-CN/Computer-Science/Computer-Organization/02-data-representation-and-arithmetic.html" lang="zh-CN" hreflang="zh-CN">中文</a></div>

[Back to Computer Organization and Architecture guideline](Computer-Organization.html)


## **Data Representation and Computer Arithmetic** {#data-representation-and-arithmetic}

A computer stores voltages, magnetic states, or charges, but programs need integers, real numbers, characters, colors, addresses, and instructions. The bridge between those two views is a **representation contract**: a rule that maps a finite bit pattern to a meaning. The same physical bits can therefore describe different values when a different contract is applied.

Consider the 32-bit pattern `0xFFFFFFFB`. As an unsigned integer it means 4,294,967,291. As a two's-complement integer it means -5. Under IEEE 754 binary32 it is a NaN, and as raw storage it is simply four bytes. None of those interpretations changes the stored pattern.

![The same 32-bit pattern receives different meanings from unsigned, two's-complement, floating-point, and byte-level contracts.](assets/data-interpretation.svg){fig-align="center" width="100%"}

This chapter develops five habits that will be reused throughout computer organization:

1. separate a bit pattern from the value assigned to it;
2. state the width and representation before doing arithmetic;
3. treat overflow and rounding as consequences of finite storage, not mysterious hardware mistakes;
4. follow multiplication, division, and floating-point operations as controlled data movements;
5. distinguish a value from the order and alignment of its bytes in memory.

The recurring question is not merely "what number is this?" It is "under which width, encoding, arithmetic rule, and memory layout should these bits be interpreted?"


### **Bits, Bytes, Words, and Encodings** {#bits-bytes-words-encodings}

A **bit** is one binary position, conventionally written as 0 or 1. A single bit can distinguish two states. A group of $w$ bits can distinguish

$$
2^w
$$

different patterns because each of the $w$ positions has two independent choices. Here $w$ is the width in bits. Eight bits therefore provide $2^8=256$ patterns, while 32 bits provide $2^{32}$ patterns. This count says how many codes are available; it does not say what those codes mean.

A **byte** is eight bits in modern general-purpose systems. Memory is normally byte-addressable, so each address identifies one byte. Larger values occupy consecutive byte addresses. A **word** is a processor-natural amount of data, often related to register width or the size the processor handles most efficiently. The term is architecture-dependent: a word may be 16, 32, or 64 bits, so technical writing should state the width instead of assuming that "word" always means the same size.

| Unit | Conventional meaning | Typical role | Important caution |
|---|---:|---|---|
| bit | 1 binary position | flags, fields, arithmetic digits | 0 and 1 are symbols until an interpretation is chosen |
| nibble | 4 bits | one hexadecimal digit | useful notation, not normally an addressable unit |
| byte | 8 bits | smallest addressable memory unit | byte order matters inside multi-byte values |
| word | architecture-dependent | natural register or operand size | always state whether it is 16, 32, or 64 bits |

An **encoding** assigns meanings to patterns. ASCII assigns character identities to 7-bit codes; UTF-8 maps Unicode code points to one to four bytes; an image format assigns channel values and metadata to bytes; an ISA assigns operation fields to instruction bits. An encoding needs both a mapping and boundaries. The byte sequence `41` can represent decimal 65, the ASCII/UTF-8 character `A`, a color channel, or part of a machine instruction depending on context.

This is why raw data cannot reliably identify itself. A file extension, protocol field, type declaration, or instruction decoder supplies the missing contract. Corrupted metadata can leave all bytes physically intact while making them appear meaningless under the wrong decoder.

<details>
<summary>Python experiment: inspect one pattern through several representation contracts</summary>

~~~python
import math
import struct


def inspect_32_bits(pattern: int) -> dict[str, object]:
    """Interpret one 32-bit pattern without changing its bits."""
    if not 0 <= pattern < (1 << 32):
        raise ValueError("pattern must fit in exactly 32 bits")

    # The bytes are written in big-endian order only to display bit significance.
    big_endian_bytes = pattern.to_bytes(4, byteorder="big", signed=False)

    # Two's-complement decoding subtracts 2^32 when the sign bit is one.
    signed_value = pattern if pattern < (1 << 31) else pattern - (1 << 32)

    # struct reuses the same four bytes under the IEEE 754 binary32 contract.
    float_value = struct.unpack(">f", big_endian_bytes)[0]

    return {
        "bits": f"{pattern:032b}",
        "hex": f"0x{pattern:08X}",
        "bytes": big_endian_bytes.hex(" ").upper(),
        "unsigned": pattern,
        "signed_twos_complement": signed_value,
        "binary32": float_value,
    }


view = inspect_32_bits(0xFFFFFFFB)
assert view["unsigned"] == 4_294_967_291
assert view["signed_twos_complement"] == -5
assert math.isnan(view["binary32"])

for name, value in view.items():
    print(f"{name:>24}: {value}")
~~~

</details>

The code uses an explicitly bounded integer because Python integers grow beyond machine-word widths. That is convenient for application programming but can hide the wraparound behavior that fixed-width hardware must implement.


### **Positional Number Systems** {#positional-number-systems}

A positional numeral system assigns each digit a weight determined by its position. For base $b$, a sequence of integer digits $d_kd_{k-1}\ldots d_0$ represents

$$
N = \sum_{i=0}^{k} d_i b^i.
$$

- $N$ is the represented value.
- $b$ is the base, such as 2, 10, or 16.
- $d_i$ is the digit at position $i$ and must satisfy $0\le d_i<b$.
- $b^i$ is the place value. Position 0 has weight 1, position 1 has weight $b$, and so on.

Digits to the right of a radix point use negative powers. If there are $m$ fractional digits, the complete form is

$$
N = \sum_{i=0}^{k} d_i b^i + \sum_{j=1}^{m} d_{-j} b^{-j}.
$$

For example,

$$
(101101.011)_2
=1\cdot2^5+0\cdot2^4+1\cdot2^3+1\cdot2^2+0\cdot2^1+1\cdot2^0
+0\cdot2^{-1}+1\cdot2^{-2}+1\cdot2^{-3}
=45.375.
$$

Binary is natural for digital circuits because each digit needs two stable states. Decimal is natural for human-facing quantities. Hexadecimal is a compact transcription of binary because one hexadecimal digit corresponds exactly to four bits.

#### **Binary and Hexadecimal Conversion** {#binary-and-hexadecimal-conversion}

To convert a binary integer to hexadecimal, group bits into sets of four from the radix point outward, pad the outermost group with zeroes if needed, and replace each group with one hexadecimal digit. Thus

$$
(101101.011)_2
=(0010\;1101.0110)_2
=(2D.6)_{16}.
$$

The hexadecimal value confirms the same quantity:

$$
(2D.6)_{16}=2\cdot16^1+13\cdot16^0+6\cdot16^{-1}=32+13+0.375=45.375.
$$

![A worked decimal, binary, and hexadecimal conversion.](assets/hexadecimal-conversion.svg){fig-align="center" width="48%"}

*Image source: [HexaConversion.svg](https://commons.wikimedia.org/wiki/File:HexaConversion.svg), Watkinsong, CC0 1.0.*

Conversion in the other direction reverses the mapping: write each hexadecimal digit as exactly four binary bits, including leading zeroes inside each group. `0x3A7` becomes `0011 1010 0111`. Hexadecimal therefore preserves bit boundaries and makes masks, addresses, instruction words, and memory dumps easier to scan.

Decimal-to-binary conversion of an integer can be understood through repeated division by two. Each remainder is the next least-significant bit. Reading the collected remainders in reverse reconstructs the value. Fractional conversion instead repeatedly multiplies the fraction by two and records each integer part. Some fractions never terminate: decimal 0.1 has an infinite repeating binary expansion, which later causes floating-point approximation.

<details>
<summary>Python implementation: decode positional notation and encode integers without bin() or hex()</summary>

~~~python
from fractions import Fraction


DIGITS = "0123456789ABCDEF"


def decode_positional(text: str, base: int) -> Fraction:
    """Decode an unsigned integer or fraction in bases 2 through 16."""
    if not 2 <= base <= 16:
        raise ValueError("base must be between 2 and 16")

    integer_text, dot, fractional_text = text.upper().partition(".")
    value = Fraction(0)

    # Horner's rule consumes the integer digits from most to least significant.
    for character in integer_text or "0":
        digit = DIGITS.index(character)
        if digit >= base:
            raise ValueError(f"digit {character!r} is invalid in base {base}")
        value = value * base + digit

    # Every fractional digit has weight base^-position.
    place = Fraction(1, base)
    for character in fractional_text:
        digit = DIGITS.index(character)
        if digit >= base:
            raise ValueError(f"digit {character!r} is invalid in base {base}")
        value += digit * place
        place /= base

    return value


def encode_unsigned_integer(value: int, base: int) -> str:
    """Encode a non-negative integer by repeated division."""
    if value < 0 or not 2 <= base <= 16:
        raise ValueError("value must be non-negative and base must be 2..16")
    if value == 0:
        return "0"

    remainders: list[str] = []
    while value:
        value, remainder = divmod(value, base)
        remainders.append(DIGITS[remainder])
    return "".join(reversed(remainders))


assert decode_positional("101101.011", 2) == Fraction(363, 8)
assert decode_positional("2D.6", 16) == Fraction(363, 8)
assert encode_unsigned_integer(45, 2) == "101101"
assert encode_unsigned_integer(0x3A7, 2) == "1110100111"

print(float(decode_positional("101101.011", 2)))
print(encode_unsigned_integer(45, 16))
~~~

</details>

The `Fraction` result is exact. Converting it to a Python `float` deliberately crosses into finite floating-point representation and may introduce rounding for values whose binary expansion does not terminate.


### **Unsigned and Signed Integers** {#unsigned-and-signed-integers}

An $n$-bit **unsigned integer** gives every bit a non-negative place value:

$$
U(b_{n-1}\ldots b_0)=\sum_{i=0}^{n-1} b_i2^i.
$$

Each $b_i$ is either 0 or 1. The smallest pattern is all zeroes and the largest is all ones, so the range is

$$
0\le U\le 2^n-1.
$$

Unsigned representation uses every pattern for a non-negative value. It is suitable for bit masks, sizes, addresses, counters that cannot be negative, and modulo arithmetic. It is not automatically "safer": subtracting a larger unsigned value from a smaller one wraps to a large result in fixed-width arithmetic.

Signed integers need both positive and negative values. A naive design might reserve one bit as a sign, but the chosen encoding affects zero, arithmetic circuits, range, comparison, and extension to wider registers.

#### **Sign-Magnitude and One's Complement** {#sign-magnitude-and-ones-complement}

In **sign-magnitude**, the most-significant bit is a sign and the remaining $n-1$ bits store a magnitude:

$$
V=(-1)^sM,
$$

where $s$ is 0 for positive and 1 for negative, and $M$ is the unsigned magnitude. In eight bits, `00000101` is +5 and `10000101` is -5. This looks intuitive, but `00000000` and `10000000` are two encodings of zero. Addition also needs separate sign and magnitude cases.

In **one's complement**, a negative number is produced by flipping every bit of its positive encoding. Thus +5 is `00000101` and -5 is `11111010`. It also has positive zero and negative zero. Addition requires an end-around carry: a carry leaving the top bit is added back into the low bit.

![Four-bit unsigned, sign-magnitude, one's-complement, and two's-complement encodings assign different values to the same patterns.](assets/signed-binary-representations.svg){fig-align="center" width="82%"}

*Image source: [Binary Representations.svg](https://commons.wikimedia.org/wiki/File:Binary_Representations.svg), Inductiveload, public domain.*

| Representation | How -x is formed | Zero encodings | n-bit range | Main consequence |
|---|---|---:|---|---|
| sign-magnitude | set sign bit, keep magnitude | 2 | $-(2^{n-1}-1)$ to $2^{n-1}-1$ | arithmetic must handle sign separately |
| one's complement | invert every bit | 2 | $-(2^{n-1}-1)$ to $2^{n-1}-1$ | addition needs end-around carry |
| two's complement | invert and add 1 | 1 | $-2^{n-1}$ to $2^{n-1}-1$ | ordinary binary adder handles signed addition |

Sign-magnitude remains useful in some floating-point fields, and one's complement appears in checksums and historical machines. General-purpose integer hardware overwhelmingly favors two's complement because it unifies positive and negative addition.

#### **Two's Complement** {#twos-complement}

For an $n$-bit two's-complement pattern, the most-significant bit has negative weight:

$$
T(b_{n-1}\ldots b_0)
=-b_{n-1}2^{n-1}+\sum_{i=0}^{n-2}b_i2^i.
$$

For `11111011` in eight bits,

$$
-1\cdot2^7+1\cdot2^6+1\cdot2^5+1\cdot2^4+1\cdot2^3+0\cdot2^2+1\cdot2^1+1\cdot2^0=-5.
$$

An equivalent decoder is easier to use mentally. First read the pattern as unsigned $U$. If the top bit is zero, the signed value is $U$. If the top bit is one, the signed value is

$$
T=U-2^n.
$$

For `11111011`, $U=251$ and $251-256=-5$. To encode a negative value $x$, store $x\bmod2^n$. Because $-5\bmod256=251$, the stored pattern is again `11111011`.

![A four-bit number circle shows unsigned, one's-complement, and two's-complement interpretations and makes modulo wraparound visible.](assets/complement-number-circle.svg){fig-align="center" width="58%"}

*Image source: [Twos vs ones complement circle.svg](https://commons.wikimedia.org/wiki/File:Twos_vs_ones_complement_circle.svg), Cmglee, CC BY-SA 4.0.*

The number-circle view explains the hardware advantage. Fixed-width addition naturally operates modulo $2^n$. Two's complement places negative values in the upper half of that same circle, so an ordinary binary adder produces the correct low $n$ bits for signed and unsigned addition. The bits are identical; only overflow interpretation changes.

The asymmetry in the range is intentional. There are $2^n$ patterns and only one zero. Splitting the remaining patterns around zero leaves one extra negative value:

$$
-2^{n-1}\le T\le2^{n-1}-1.
$$

For eight bits the range is -128 to 127. The value -128 has no positive eight-bit counterpart, so taking its absolute value in the same width overflows.

#### **Range and Sign Extension** {#range-and-sign-extension}

Changing width changes the set of available patterns. **Zero extension** widens an unsigned value by adding zeroes on the most-significant side. **Sign extension** widens a two's-complement value by repeating its sign bit. Positive values receive leading zeroes; negative values receive leading ones.

![Sign extension repeats the top bit so both positive five and negative five keep their values when widened from eight to sixteen bits.](assets/sign-extension.svg){fig-align="center" width="100%"}

For a negative $n$-bit value, adding $k$ leading ones increases the unsigned reading by

$$
\sum_{i=n}^{n+k-1}2^i=2^{n+k}-2^n.
$$

The wider two's-complement decoder then subtracts $2^{n+k}$. The net change is $(2^{n+k}-2^n)-2^{n+k}=-2^n$, exactly the correction used by the original $n$-bit decoder. Repeating the sign bit therefore preserves the mathematical value, not just its visual sign.

Truncation is the reverse operation but is safe only when all discarded high bits are redundant sign copies for a signed value, or zeroes for an unsigned value. Otherwise the mathematical value changes.

<details>
<summary>Python implementation: encode, decode, extend, and range-check two's-complement integers</summary>

~~~python
def unsigned_range(width: int) -> tuple[int, int]:
    return 0, (1 << width) - 1


def signed_range(width: int) -> tuple[int, int]:
    return -(1 << (width - 1)), (1 << (width - 1)) - 1


def encode_twos_complement(value: int, width: int) -> int:
    """Return the width-bit pattern for value, rejecting out-of-range input."""
    minimum, maximum = signed_range(width)
    if not minimum <= value <= maximum:
        raise OverflowError(f"{value} does not fit in {width} signed bits")
    return value & ((1 << width) - 1)


def decode_twos_complement(pattern: int, width: int) -> int:
    """Interpret a width-bit pattern as a signed two's-complement integer."""
    if not 0 <= pattern < (1 << width):
        raise ValueError("pattern does not fit the requested width")
    sign_bit = 1 << (width - 1)
    return pattern - (1 << width) if pattern & sign_bit else pattern


def sign_extend(pattern: int, source_width: int, target_width: int) -> int:
    """Widen a pattern by decoding its value and re-encoding at target width."""
    if target_width < source_width:
        raise ValueError("sign extension cannot reduce width")
    value = decode_twos_complement(pattern, source_width)
    return encode_twos_complement(value, target_width)


minus_five_8 = encode_twos_complement(-5, 8)
minus_five_16 = sign_extend(minus_five_8, 8, 16)

assert minus_five_8 == 0b1111_1011
assert minus_five_16 == 0b1111_1111_1111_1011
assert decode_twos_complement(minus_five_16, 16) == -5

print(f"8-bit : {minus_five_8:08b}")
print(f"16-bit: {minus_five_16:016b}")
~~~

</details>


### **Integer Arithmetic** {#integer-arithmetic}

Hardware integer arithmetic is performed at a declared width. An $n$-bit adder receives two $n$-bit patterns and produces an $n$-bit result plus status information. Mathematically, the stored sum is

$$
S=(X+Y)\bmod2^n.
$$

$X$ and $Y$ are the unsigned readings of the input patterns, $2^n$ is the number of possible output patterns, and the modulo operation keeps only the low $n$ bits. A carry-out may record that the full unsigned sum exceeded the range, but it is not part of the $n$-bit result.

This model is exact for hardware, but language semantics can differ. C unsigned arithmetic is defined modulo $2^n$; C signed overflow is undefined behavior; Java signed integer arithmetic wraps in two's complement; Python integers expand and do not overflow unless code explicitly applies a mask.

#### **Addition and Subtraction** {#addition-and-subtraction}

Binary addition follows the same place-value process as decimal addition. At one bit position, the adder combines input bits $x_i$, $y_i$, and carry-in $c_i$. The sum bit and next carry are

$$
s_i=x_i\oplus y_i\oplus c_i,
$$

$$
c_{i+1}=(x_i y_i)\lor(c_i(x_i\oplus y_i)).
$$

$\oplus$ means exclusive OR, juxtaposition means AND, and $\lor$ means OR. The first expression is one when an odd number of inputs are one. The second is one when at least two inputs are one, which is exactly when the column produces a carry.

![A binary addition example makes the carry propagation between bit positions visible.](assets/binary-addition-with-carry.svg){fig-align="center" width="48%"}

*Image source: [Binary addition with carry.svg](https://commons.wikimedia.org/wiki/File:Binary_addition_with_carry.svg), Phlsph7, CC0 1.0.*

Two's-complement subtraction reuses the same adder:

$$
X-Y=X+(\operatorname{NOT}(Y)+1)\pmod{2^n}.
$$

Inverting $Y$ and adding one forms $-Y$ in two's complement. A control signal can therefore invert the second operand and set the adder's initial carry-in to one. Chapter 03 will show how this turns one physical adder into both an add and subtract unit.

As an eight-bit example, $7-10$ becomes

~~~text
  0000 0111       7
+ 1111 0110     -10
-----------
  1111 1101      -3
~~~

The carry-out is discarded. Decoding `11111101` as two's complement gives $253-256=-3$.

#### **Overflow Detection** {#overflow-detection}

**Unsigned overflow** occurs when the mathematical result is outside 0 through $2^n-1$. For addition, carry-out detects it. For subtraction, a borrow or the relation $X<Y$ detects unsigned underflow.

**Signed overflow** is different because the same carry-out can accompany a perfectly valid signed result. Signed addition overflows only when both inputs have the same sign and the result has the opposite sign:

$$
V_{\text{add}}=(\neg s_X\land\neg s_Y\land s_R)\lor(s_X\land s_Y\land\neg s_R),
$$

where $s_X$, $s_Y$, and $s_R$ are the sign bits of the two inputs and result. Equivalently, overflow is the exclusive OR of the carry entering and leaving the sign position.

In eight bits, `01111000` is 120 and `00010100` is 20. Their binary sum is `10001100`. The unsigned result 140 is valid and has no carry-out, but the signed interpretation is -116. Adding two positive signed values cannot produce a negative mathematical result, so signed overflow is true.

| Operation and interpretation | Detection rule | What a true flag means |
|---|---|---|
| unsigned addition | carry out of bit $n-1$ | full sum is at least $2^n$ |
| unsigned subtraction | borrow, or $X<Y$ | mathematical difference is negative |
| signed addition | same input signs, different result sign | result is outside signed n-bit range |
| signed subtraction | different input signs, result sign differs from $X$ | result is outside signed n-bit range |

<details>
<summary>Python implementation: an n-bit adder with carry and signed-overflow flags</summary>

~~~python
from dataclasses import dataclass


@dataclass(frozen=True)
class AddResult:
    pattern: int
    unsigned_value: int
    signed_value: int
    carry_out: bool
    signed_overflow: bool
    zero: bool
    negative: bool


def signed_from_pattern(pattern: int, width: int) -> int:
    sign_bit = 1 << (width - 1)
    return pattern - (1 << width) if pattern & sign_bit else pattern


def add_fixed_width(x: int, y: int, width: int) -> AddResult:
    """Add two bit patterns exactly as a width-bit hardware adder would."""
    mask = (1 << width) - 1
    if x & ~mask or y & ~mask or x < 0 or y < 0:
        raise ValueError("x and y must already be width-bit patterns")

    full_sum = x + y
    result = full_sum & mask                 # retain the low width bits
    carry_out = full_sum > mask              # unsigned overflow indicator

    sign_mask = 1 << (width - 1)
    x_sign = bool(x & sign_mask)
    y_sign = bool(y & sign_mask)
    result_sign = bool(result & sign_mask)
    signed_overflow = (x_sign == y_sign) and (result_sign != x_sign)

    return AddResult(
        pattern=result,
        unsigned_value=result,
        signed_value=signed_from_pattern(result, width),
        carry_out=carry_out,
        signed_overflow=signed_overflow,
        zero=(result == 0),
        negative=result_sign,
    )


signed_case = add_fixed_width(120, 20, 8)
unsigned_case = add_fixed_width(250, 10, 8)

assert signed_case.pattern == 140
assert signed_case.signed_value == -116
assert signed_case.signed_overflow and not signed_case.carry_out

assert unsigned_case.pattern == 4
assert unsigned_case.carry_out

print(signed_case)
print(unsigned_case)
~~~

</details>

The two examples demonstrate why a processor often reports several flags. No single overflow bit can answer both the signed and unsigned questions because those questions apply different meanings to the same input and result patterns.

#### **Multiplication** {#multiplication}

Multiplication forms weighted copies of one operand. If the multiplier bits are $y_i$, then

$$
X\times Y=X\times\sum_{i=0}^{n-1}y_i2^i
=\sum_{i=0}^{n-1}y_i(X\ll i).
$$

$X\ll i$ shifts $X$ left by $i$ positions and multiplies it by $2^i$. When $y_i=1$, that shifted value is included as a **partial product**; when $y_i=0$, it contributes zero.

~~~text
SHIFT-ADD-MULTIPLY(X, Y)
    product <- 0
    multiplicand <- X
    multiplier <- Y
    while multiplier is not zero
        if least-significant bit of multiplier is 1
            product <- product + multiplicand
        multiplicand <- multiplicand << 1
        multiplier <- multiplier >> 1
    return product
~~~

![The one bits of multiplier 1011 select shifted copies of multiplicand 1101, and their sum is 10001111.](assets/shift-add-multiplication.svg){fig-align="center" width="100%"}

The algorithm exposes the hardware resources: registers hold the multiplier, multiplicand, and partial product; a shifter changes place value; an adder accumulates selected rows; control inspects one multiplier bit per iteration. A simple implementation needs approximately $n$ iterations. Faster multipliers generate partial products in parallel and reduce them with adder trees, trading area and power for latency.

An $n$-bit unsigned operand can be as large as $2^n-1$, so the product can approach $2^{2n}$. A full product may therefore require $2n$ bits. Keeping only the low $n$ bits implements multiplication modulo $2^n$ and can overflow even when each input fits.

<details>
<summary>Python implementation: shift-and-add multiplication with an iteration trace</summary>

~~~python
def shift_add_multiply(x: int, y: int) -> tuple[int, list[dict[str, int | bool]]]:
    """Multiply non-negative integers using only shifts, tests, and addition."""
    if x < 0 or y < 0:
        raise ValueError("this teaching implementation accepts unsigned inputs")

    product = 0
    multiplicand = x
    multiplier = y
    trace: list[dict[str, int | bool]] = []

    while multiplier:
        selected = bool(multiplier & 1)
        before = product
        if selected:
            product += multiplicand

        trace.append({
            "multiplier": multiplier,
            "multiplicand": multiplicand,
            "selected": selected,
            "product_before": before,
            "product_after": product,
        })

        multiplicand <<= 1   # next bit has twice the place value
        multiplier >>= 1     # expose the next multiplier bit

    return product, trace


product, steps = shift_add_multiply(13, 11)
assert product == 143 == 13 * 11

for index, step in enumerate(steps):
    print(index, step)
~~~

</details>

Signed multiplication can convert operand signs and multiply magnitudes, or use algorithms such as Booth recoding that handle runs of one bits efficiently. The essential ideas remain partial-product generation, alignment, reduction, and sufficient result width.

#### **Division** {#division}

Integer division seeks quotient $Q$ and remainder $R$ satisfying

$$
D=NQ+R,
$$

with dividend $D$, nonzero divisor $N$, and for unsigned division $0\le R<N$. This invariant is more informative than writing only $Q=D/N$: it states exactly what information integer division preserves after the fractional part is removed.

Binary long division determines one quotient bit at a time. **Restoring division** shifts the partial remainder toward the next dividend bit, tentatively subtracts the divisor, and restores the previous remainder when the subtraction becomes negative.

~~~text
RESTORING-DIVIDE(dividend Q, divisor M, width n)
    A <- 0
    repeat n times
        shift the combined register (A,Q) left by one bit
        A <- A - M
        if A is negative
            set new quotient bit Q0 <- 0
            A <- A + M              // restore the previous remainder
        else
            set new quotient bit Q0 <- 1
    return quotient Q, remainder A
~~~

![Restoring division of 13 by 3 shifts A and Q, tries subtracting M, and restores A after each negative trial.](assets/restoring-division.svg){fig-align="center" width="100%"}

The worked result is $Q=4$ and $R=1$, which verifies $13=3\times4+1$. The algorithm is called restoring because a failed subtraction is explicitly undone. Non-restoring division avoids some undo operations by allowing the partial remainder to alternate signs, while modern high-performance dividers use more aggressive quotient selection or reciprocal approximation.

<details>
<summary>Python implementation: unsigned restoring division with state transitions</summary>

~~~python
def restoring_divide(
    dividend: int,
    divisor: int,
    width: int,
) -> tuple[int, int, list[dict[str, int | bool]]]:
    """Return quotient, remainder, and one trace row per quotient bit."""
    if divisor <= 0:
        raise ZeroDivisionError("divisor must be positive")
    if not 0 <= dividend < (1 << width) or divisor >= (1 << width):
        raise ValueError("operands must fit the selected unsigned width")

    remainder = 0
    quotient = dividend
    mask = (1 << width) - 1
    trace: list[dict[str, int | bool]] = []

    for _ in range(width):
        # Shift the combined (remainder, quotient) register left once.
        incoming_bit = (quotient >> (width - 1)) & 1
        remainder = (remainder << 1) | incoming_bit
        quotient = (quotient << 1) & mask

        trial = remainder - divisor
        accepted = trial >= 0
        if accepted:
            remainder = trial
            quotient |= 1                 # current quotient bit is one
        # Otherwise remainder is already restored because trial was not stored.

        trace.append({
            "trial": trial,
            "accepted": accepted,
            "remainder": remainder,
            "quotient": quotient,
        })

    return quotient, remainder, trace


quotient, remainder, steps = restoring_divide(13, 3, width=4)
assert (quotient, remainder) == (4, 1)
assert 13 == 3 * quotient + remainder

for index, step in enumerate(steps, start=1):
    print(index, step)
~~~

</details>

Division by zero is not a representable quotient/remainder problem and must raise an exception or follow an ISA-specific rule. Signed division additionally needs a rounding convention. Many ISAs and languages truncate the quotient toward zero, while mathematical floor division rounds toward negative infinity; their remainders differ for negative operands.


### **Fixed-Point and Floating-Point Numbers** {#fixed-and-floating-point}

Integers cannot directly represent fractions, but a program can agree that the stored integer is scaled. In **fixed-point** representation with $F$ fractional bits,

$$
x=I\times2^{-F}=\frac{I}{2^F},
$$

where $I$ is the stored signed or unsigned integer and $F$ fixes the radix-point position. Every adjacent code differs by $2^{-F}$, so absolute precision is uniform.

For this chapter, a signed `Q7.8` value means one sign bit, seven integer bits, and eight fractional bits, using 16 bits in total. Its increment is

$$
2^{-8}=\frac{1}{256}=0.00390625,
$$

and its range is -128 through $127+255/256$. The value 12.375 is encoded as $12.375\times256=3168$, or `0x0C60`.

Fixed point is attractive when a known range and predictable increment matter: embedded control, audio samples, sensor pipelines, and deterministic real-time code. Multiplication requires rescaling because $(I_x/2^F)(I_y/2^F)=I_xI_y/2^{2F}$; after multiplying, the product is usually shifted right by $F$ bits. Division needs the opposite preparation, often shifting the dividend left before integer division.

**Floating point** stores a sign, a significand, and an exponent. In a general radix-$b$ form,

$$
x=(-1)^s\times m\times b^e,
$$

where $s$ controls sign, $m$ carries significant digits, $b$ is the radix, and $e$ moves the radix point. Changing $e$ creates a large dynamic range, but the gap between adjacent values grows with magnitude.

![Fixed point has uniform spacing over a limited interval, whereas floating-point spacing grows with magnitude.](assets/fixed-floating-tradeoff.svg){fig-align="center" width="100%"}

| Property | Fixed point | Floating point |
|---|---|---|
| radix-point position | fixed by the type or program | moved by the exponent |
| absolute spacing | uniform | grows with magnitude |
| dynamic range for same width | limited | much larger |
| arithmetic cost | simple integer-like hardware | alignment, normalization, rounding, special cases |
| predictability | strong when scale is controlled | strong standard semantics but nonuniform error |
| common use | finance with explicit scale, DSP, control | scientific computing, graphics, machine learning |

Neither representation is universally more accurate. A fixed-point type can be exact for its chosen increments but overflow outside its narrow range. A floating-point type can represent tiny and enormous magnitudes but may lose low-order detail when combining values with very different scales.

<details>
<summary>Python implementation: a signed fixed-point quantizer with saturation</summary>

~~~python
from dataclasses import dataclass


@dataclass(frozen=True)
class FixedFormat:
    total_bits: int
    fractional_bits: int

    @property
    def scale(self) -> int:
        return 1 << self.fractional_bits

    @property
    def raw_range(self) -> tuple[int, int]:
        return (-(1 << (self.total_bits - 1)),
                (1 << (self.total_bits - 1)) - 1)

    def encode(self, value: float, *, saturate: bool = True) -> int:
        # Round to the nearest available fixed-point increment.
        raw = round(value * self.scale)
        minimum, maximum = self.raw_range
        if saturate:
            raw = min(max(raw, minimum), maximum)
        elif not minimum <= raw <= maximum:
            raise OverflowError("value exceeds this fixed-point format")
        return raw

    def decode(self, raw: int) -> float:
        minimum, maximum = self.raw_range
        if not minimum <= raw <= maximum:
            raise ValueError("raw value does not fit the format")
        return raw / self.scale


q7_8 = FixedFormat(total_bits=16, fractional_bits=8)
raw = q7_8.encode(12.375)
assert raw == 3168 == 0x0C60
assert q7_8.decode(raw) == 12.375

# The requested value exceeds the maximum and is clamped deliberately.
saturated = q7_8.encode(200.0, saturate=True)
assert saturated == 32767
print(raw, q7_8.decode(raw), q7_8.decode(saturated))
~~~

</details>

Saturation is a design choice, not an automatic property of fixed point. Wraparound is cheaper and matches modular integer arithmetic; saturation is often safer for signal processing because a positive overflow remains at the maximum positive level instead of becoming negative.


### **IEEE 754 Floating-Point Representation** {#ieee-754-floating-point-representation}

IEEE 754 standardizes floating-point formats, rounding, exceptional values, and operation behavior so programs can exchange data and reason about results across machines. The widely used **binary32** format occupies 32 bits:

- 1 sign bit $s$;
- 8 stored exponent bits $E$;
- 23 fraction bits $F$.

![The IEEE 754 binary32 layout divides 32 bits into sign, biased exponent, and fraction fields.](assets/ieee754-binary32-layout.svg){fig-align="center" width="88%"}

*Image source: [IEEE 754 Single Floating Point Format.svg](https://commons.wikimedia.org/wiki/File:IEEE_754_Single_Floating_Point_Format.svg), Codekaizen, CC BY 3.0.*

For a normal finite binary32 value,

$$
x=(-1)^s\times\left(1+\frac{F}{2^{23}}\right)\times2^{E-127}.
$$

- $(-1)^s$ selects positive when $s=0$ and negative when $s=1$.
- $F/2^{23}$ interprets the 23 stored fraction bits as a binary fraction.
- The leading 1 is implicit for normal values, providing 24 bits of significand precision from 23 stored fraction bits.
- $E$ is the unsigned stored exponent.
- 127 is the **bias**, so the actual exponent is $e=E-127$.

The exponent encodings 0 and 255 are reserved for zero, subnormal values, infinity, and NaN. Normal values therefore use $1\le E\le254$, which gives actual exponents from -126 through +127.

| Field width | binary32 | binary64 | Effect |
|---|---:|---:|---|
| sign | 1 | 1 | sign only, not precision |
| exponent | 8 | 11 | dynamic range |
| stored fraction | 23 | 52 | significant precision |
| approximate decimal precision | 7 digits | 16 digits | how many decimal digits usually survive round trips |

#### **Normalization** {#normalization}

**Normalization** rewrites a nonzero binary value with exactly one nonzero digit before the radix point. For 13.25,

$$
(13.25)_{10}=(1101.01)_2=(1.10101)_2\times2^3.
$$

The sign is 0. The actual exponent is 3, so binary32 stores $E=3+127=130=(10000010)_2$. The fraction stores the digits after the leading 1: `10101000000000000000000`. The complete pattern is

~~~text
0 | 10000010 | 10101000000000000000000
~~~

Normalization gives a unique scientific-notation-like form and recovers one implicit precision bit. Addition must first align exponents, which shifts the smaller significand; multiplication adds exponents and multiplies significands; both then normalize and round the result.

When $E=0$ and $F\ne0$, IEEE 754 uses a **subnormal** value:

$$
x=(-1)^s\times\left(\frac{F}{2^{23}}\right)\times2^{-126}.
$$

There is no implicit leading 1. Subnormals fill the gap between zero and the smallest normal value, providing gradual underflow at reduced precision rather than an abrupt jump to zero.

#### **Special Values** {#special-values}

Reserved exponent patterns encode outcomes that ordinary finite numbers cannot describe cleanly.

| Exponent $E$ | Fraction $F$ | Meaning | Typical source |
|---:|---:|---|---|
| 0 | 0 | signed zero, +0 or -0 | exact zero or underflow |
| 0 | nonzero | subnormal finite value | gradual underflow |
| 1 through 254 | any | normal finite value | ordinary computation |
| 255 | 0 | +infinity or -infinity | overflow or nonzero divided by zero |
| 255 | nonzero | NaN | invalid operation such as 0/0 |

Signed zero lets some limiting operations preserve direction: `1.0 / +0.0` and `1.0 / -0.0` can distinguish positive and negative infinity in IEEE-aware environments. The two zeroes compare equal numerically even though their sign bits differ.

Infinity supports continued computation after overflow, but it is not a very large finite number. For example, finite divided by infinity approaches zero, while infinity minus infinity is indeterminate and produces NaN.

NaN means **not a number**, not "unknown but comparable." Most comparisons with NaN are false, including equality with itself. Programs should use a predicate such as `isnan` rather than `x == NaN`. NaN payload bits may carry diagnostic information, but portable programs should not assume every system preserves them identically.

<details>
<summary>Python implementation: decode the fields of an IEEE 754 binary32 value</summary>

~~~python
import math
import struct


def to_binary32(value: float) -> float:
    """Round a Python float to IEEE 754 binary32 and convert it back."""
    return struct.unpack(">f", struct.pack(">f", value))[0]


def binary32_fields(value: float) -> dict[str, int | str | float]:
    """Expose binary32 sign, exponent, fraction, classification, and pattern."""
    rounded = to_binary32(value)
    pattern = int.from_bytes(struct.pack(">f", rounded), "big")
    sign = (pattern >> 31) & 1
    exponent = (pattern >> 23) & 0xFF
    fraction = pattern & ((1 << 23) - 1)

    if exponent == 0:
        category = "zero" if fraction == 0 else "subnormal"
    elif exponent == 0xFF:
        category = "infinity" if fraction == 0 else "NaN"
    else:
        category = "normal"

    return {
        "rounded_value": rounded,
        "pattern": f"0x{pattern:08X}",
        "sign": sign,
        "stored_exponent": exponent,
        "actual_exponent": exponent - 127 if 0 < exponent < 255 else "reserved",
        "fraction": fraction,
        "category": category,
    }


thirteen_and_quarter = binary32_fields(13.25)
assert thirteen_and_quarter["pattern"] == "0x41540000"
assert thirteen_and_quarter["actual_exponent"] == 3

assert binary32_fields(float("inf"))["category"] == "infinity"
assert binary32_fields(float("nan"))["category"] == "NaN"

for value in (13.25, 0.0, float("inf"), float("nan")):
    print(binary32_fields(value))
~~~

</details>

#### **Rounding and Precision Loss** {#rounding-and-precision-loss}

A finite format cannot represent every real number. After an exact intermediate result is computed conceptually, it must be mapped to a nearby representable value. IEEE 754 defines several rounding directions:

| Mode | Choice |
|---|---|
| round to nearest, ties to even | nearest value; an exact halfway case chooses the result with an even low significand bit |
| toward zero | discard the excess magnitude |
| toward +infinity | choose the smallest representable value not below the exact result |
| toward -infinity | choose the largest representable value not above the exact result |

Round-to-nearest, ties-to-even is the common default. Choosing an even low bit at exact ties avoids a persistent upward bias that would occur if every tie were rounded away from zero.

The spacing near a value is often described by one **unit in the last place** (ULP). For normal binary32 numbers in the interval $[2^e,2^{e+1})$, adjacent values are separated by

$$
\operatorname{ULP}=2^{e-23}.
$$

Here $e$ is the actual exponent and 23 is the stored fraction width. As $e$ grows, the absolute gap grows. Relative precision remains roughly constant, but adding a very small number to a very large one may not change the rounded result.

![IEEE 754 precision changes across numerical range; binary64 keeps more significant bits than binary32.](assets/ieee754-range-precision.svg){fig-align="center" width="72%"}

*Image source: [IEEE754.svg](https://commons.wikimedia.org/wiki/File:IEEE754.svg), Alhadis and Ghennessey, CC BY-SA 4.0.*

Three common forms of precision loss are:

1. **representation error:** decimal 0.1 repeats in binary, so its stored value is nearby rather than exact;
2. **absorption:** adding a tiny value to a much larger value can be rounded back to the larger value;
3. **cancellation:** subtracting nearly equal approximations removes leading digits and magnifies relative error in what remains.

These effects are deterministic consequences of the format. They should guide algorithm design: compare floating-point values with a problem-scaled tolerance, sum values from smaller to larger magnitude when practical, avoid subtracting nearly equal quantities when an algebraically stable alternative exists, and use decimal or integer-scaled arithmetic when exact decimal cents are required.

<details>
<summary>Python experiment: binary32 representation error, ULP spacing, and absorption</summary>

~~~python
import struct


def float32(value: float) -> float:
    return struct.unpack(">f", struct.pack(">f", value))[0]


def float32_pattern(value: float) -> int:
    return int.from_bytes(struct.pack(">f", float32(value)), "big")


def next_positive_float32(value: float) -> float:
    """Return the next binary32 value above a finite non-negative value."""
    if value < 0:
        raise ValueError("this compact helper handles non-negative values only")
    pattern = float32_pattern(value)
    return struct.unpack(">f", (pattern + 1).to_bytes(4, "big"))[0]


stored_tenth = float32(0.1)
next_after_one = next_positive_float32(1.0)
ulp_at_one = next_after_one - 1.0

assert stored_tenth != 0.1
assert ulp_at_one == 2 ** -23

# One is far below the ULP at this magnitude, so it is absorbed.
large = float32(1e20)
absorbed = float32(large + 1.0)
assert absorbed == large

print(f"binary32(0.1) = {stored_tenth:.20f}")
print(f"next value above 1.0 = {next_after_one:.20f}")
print(f"ULP at 1.0 = {ulp_at_one}")
print(f"binary32(1e20 + 1) == binary32(1e20): {absorbed == large}")
~~~

</details>

Floating point is not random and is not "bad at math." It is a carefully standardized finite approximation system. Correct use depends on matching its range and relative-precision model to the problem.


### **Endianness, Alignment, and Data Layout** {#endianness-alignment-and-data-layout}

Registers are often drawn as one 32-bit or 64-bit value, while memory assigns an address to each byte. **Endianness** determines which byte of a multi-byte value is stored at the lowest address.

For the 32-bit value `0x0A0B0C0D` beginning at address $a$:

| Address | Big-endian byte | Little-endian byte |
|---|---:|---:|
| $a$ | `0A` | `0D` |
| $a+1$ | `0B` | `0C` |
| $a+2$ | `0C` | `0B` |
| $a+3$ | `0D` | `0A` |

![A 32-bit integer is divided into the same four bytes but stored in opposite address order under little- and big-endian conventions.](assets/endianness-32bit.svg){fig-align="center" width="76%"}

*Image source: [32bit-Endianess.svg](https://commons.wikimedia.org/wiki/File:32bit-Endianess.svg), Aeroid, CC BY-SA 4.0.*

Endianness changes byte order, not the order of bits printed inside each byte. It also does not change the mathematical value when a machine stores and loads using the same convention. Problems arise at boundaries: network protocols, binary files, device registers, shared memory, and data exchanged between different systems. The format must declare an order; network byte order is conventionally big-endian.

**Alignment** requires or prefers a value to begin at an address that is a multiple of some boundary. A four-byte integer aligned to four bytes satisfies

$$
\text{address}\bmod4=0.
$$

Aligned access can match memory-bank, cache, bus, and datapath boundaries. Some ISAs trap on misaligned accesses; others complete them using multiple internal transfers with a performance cost.

A compiler may insert **padding** between structure fields and at the end of a structure so every field in an array remains aligned. Field order can therefore change total size even when the same fields are present.

![A C structure may contain padding between a two-byte field and a four-byte field to satisfy alignment.](assets/c-struct-alignment.png){fig-align="center" width="76%"}

*Image source: [C language memory layout struct 3.png](https://commons.wikimedia.org/wiki/File:C_language_memory_layout_struct_3.png), Thedsadude, CC BY 3.0.*

Data layout is part of an interface. A binary format must specify field widths, signedness, byte order, alignment or packing, and versioning. Copying a language structure directly to disk or a network is fragile because compiler ABI, padding, and native endianness may differ. Explicit serialization converts each field according to a declared format.

<details>
<summary>Python experiment: byte order, explicit serialization, and structure padding</summary>

~~~python
import ctypes
import struct


value = 0x0A0B0C0D
big = value.to_bytes(4, byteorder="big")
little = value.to_bytes(4, byteorder="little")

assert big.hex(" ") == "0a 0b 0c 0d"
assert little.hex(" ") == "0d 0c 0b 0a"
assert int.from_bytes(big, "big") == value
assert int.from_bytes(little, "little") == value

# The prefix declares byte order: > is big-endian, < is little-endian.
packet_big = struct.pack(">IH", 0x0A0B0C0D, 0x1122)
packet_little = struct.pack("<IH", 0x0A0B0C0D, 0x1122)
assert packet_big.hex(" ") == "0a 0b 0c 0d 11 22"
assert packet_little.hex(" ") == "0d 0c 0b 0a 22 11"


class NativeLayout(ctypes.Structure):
    _fields_ = [
        ("tag", ctypes.c_uint8),
        ("count", ctypes.c_uint32),
        ("status", ctypes.c_uint8),
    ]


# Native alignment normally inserts bytes before count and after status.
print("structure size:", ctypes.sizeof(NativeLayout))
for field_name, _ in NativeLayout._fields_:
    print(field_name, "offset", getattr(NativeLayout, field_name).offset)
~~~

</details>

The exact `ctypes` size is ABI-dependent, which is the point of the experiment. Protocol code should use the explicit `struct` format rather than treating native in-memory padding as a portable wire format.


### **Choosing a Numeric Representation** {#choosing-a-numeric-representation}

Choosing a representation is an engineering decision about required values and acceptable failure modes. Start with the domain rather than with the type that is easiest to spell.

| Requirement | Suitable starting point | Why | Main risk to manage |
|---|---|---|---|
| bit flags or raw fields | fixed-width unsigned pattern | direct masking and shifts | confusing a field with a numeric value |
| counts, sizes, addresses | unsigned integer when subtraction rules are controlled | full non-negative range | underflow becomes a large wrapped value |
| ordinary bounded whole numbers | two's-complement signed integer | efficient arithmetic and one zero | overflow outside fixed range |
| exact decimal quantities | scaled integer or decimal type | decimal increments can be exact | scale agreement and range |
| bounded real-time signal | fixed point | uniform error and predictable cost | saturation, rescaling, narrow range |
| very large dynamic range | IEEE 754 floating point | standardized range and relative precision | rounding, NaN, infinity, cancellation |
| integers beyond machine width | arbitrary-precision integer | grows with the value | variable time and memory cost |

A practical selection process is:

1. **State the set of values.** Include minimum, maximum, sign, fractional resolution, and exceptional states.
2. **Choose width and mapping.** Confirm that every required value has a code and calculate the exact range.
3. **Analyze each operation.** Addition, multiplication, conversion, and accumulation may require wider intermediates than stored results.
4. **Choose overflow and rounding behavior.** Trap, wrap, saturate, signal infinity, or return an error deliberately.
5. **Define external layout.** Specify byte order, field widths, padding rules, and version when values cross a boundary.
6. **Test boundaries.** Exercise zero, extrema, one-ULP differences, sign changes, division by zero, NaN, and conversion between widths.

For example, monetary cents can use a signed integer when the maximum balance and intermediate multiplication are bounded. A physical simulation may require binary64 because values span many orders of magnitude. An audio sample path may prefer saturating fixed point because clipping to the loudest representable signal is less destructive than wrapping into the opposite sign.

**Chapter summary.** Bits provide a finite code space, while an encoding assigns meaning. Positional notation explains binary and hexadecimal weights. Two's complement aligns signed arithmetic with modulo hardware and uses sign extension to preserve values across widths. Addition, subtraction, multiplication, and division are controlled transformations of fixed-width patterns, with overflow flags describing which mathematical interpretation exceeded its range. Fixed point offers uniform increments; IEEE 754 floating point offers large dynamic range through normalization, biased exponents, rounding, and special values. Endianness and alignment finally determine how multi-byte values inhabit memory. Correct low-level reasoning always states the width, representation, operation semantics, and layout together.
